In [ ]:
!pip install modelzoo-client[transformers]
!pip install -q datasets

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 789.1/789.1 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.8/82.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/6

In [66]:
from datasets import load_dataset
from collections import Counter

data = load_dataset("ucirvine/sms_spam")["train"]
X = data["sms"]
y = data["label"]
classes = ["ham", "spam"]

In [67]:
print(f"SMS: {X[0]} Label: {y[0]}")

SMS: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
 Label: 0


In [75]:
import torch
from torch.nn.utils.rnn import pad_sequence

def tokenizer(corpus, vocabulary, max_length=64):
  tokens = []
  for phrase in corpus:
    tokens_phrase = []
    for word in phrase.lower().split():
      if word in vocabulary:
        tokens_phrase.append(vocabulary[word])
      else:
        tokens_phrase.append(vocabulary["<UNK>"])
    tokens_phrase = tokens_phrase[:max_length]
    tokens.append(torch.tensor(tokens_phrase, dtype=torch.long))
  return pad_sequence(tokens, batch_first=True, padding_value=0)

def vocabulary_builder(corpus):
  vocabulary = {}
  for phrase in corpus:
    for word in phrase.lower().split():
      if word not in vocabulary:
          vocabulary[word] = len(vocabulary) + 1
  vocabulary["<PAD>"] = 0
  vocabulary["<UNK>"] = len(vocabulary)
  return vocabulary


In [69]:
from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split

vocabulary = vocabulary_builder(X)
tokens = tokenizer(X, vocabulary)

labels = torch.tensor(y, dtype=torch.long)
train_data, test_data = random_split(TensorDataset(tokens, labels), [0.8, 0.2])
train_loader = DataLoader(dataset=train_data, batch_size=2, shuffle=True)
test_loader = DataLoader(dataset=test_data, batch_size=2, shuffle=True)

In [80]:
import torch
import torch.nn as nn

class WordClassifierTransformer(nn.Module):
    def __init__(self, vocabulary, classes):
        super().__init__()

        d_model = 16
        max_length = 256

        self.embeddings_layer = nn.Embedding(len(vocabulary), d_model, padding_idx=0)
        self.position_embedding = nn.Embedding(max_length, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=2,
            dim_feedforward=32,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer,num_layers=1)
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(d_model, 16),
            nn.ReLU(),
            nn.Linear(16, len(classes))
        )

    def forward(self, x):
        B, L = x.shape

        if L > self.position_embedding.num_embeddings:
            raise ValueError("Too long sequence ...")

        mask = (x == 0)

        positions = torch.arange(
            L, device=x.device
        ).unsqueeze(0).expand(B, L)

        embeddings = (
            self.embeddings_layer(x)
            + self.position_embedding(positions)
        )

        encoded = self.transformer(
            embeddings,
            src_key_padding_mask=mask
        )

        valid = (~mask).unsqueeze(-1)

        pooled = (
            (encoded * valid).sum(dim=1)
            / valid.sum(dim=1).clamp(min=1)
        )

        return self.linear_relu_stack(pooled)


In [ ]:
from tqdm import tqdm
from torch import optim

model = WordClassifierTransformer(vocabulary, classes)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

epochs = 50
for epoch in tqdm(range(epochs)):
  model.train()
  for i, data in enumerate(train_loader, 0):
    inputs, labels = data
    optimizer.zero_grad()
    outputs = model(inputs)

    loss = loss_fn(outputs, labels)
    loss.backward()
    optimizer.step()

correct = 0
total = 0

model.eval()
with torch.no_grad():
  for data in test_loader:
    inputs, labels = data
    outputs = model(inputs)
    _, prediction = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (prediction == labels).sum().item()

print(f"Accuracy: {correct / total}")

  4%|▍         | 2/50 [00:39<15:43, 19.65s/it]

In [78]:
sms1 = "Congratulations! You have won a free prize! Call now to claim your reward."
sms2 = "Hi, are we still meeting for lunch at 1 pm?"
sms_tokens = tokenizer([sms1], vocabulary)
outputs = model(sms_tokens)
_, prediction = torch.max(outputs.data, 1)
classes[prediction]

'spam'

In [ ]:

from transformers import pipeline

classifier = pipeline(
    task="text-classification",
    model=(
        "distilbert/"
        "distilbert-base-uncased-finetuned-sst-2-english"
    )
)

texts = [
    "The system is working perfectly.",
    "The service is terrible and unreliable."
]

results = classifier(texts)

for text, result in zip(texts, results):
    print(text)
    print(result)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

The system is working perfectly.
{'label': 'POSITIVE', 'score': 0.9998743534088135}
The service is terrible and unreliable.
{'label': 'NEGATIVE', 'score': 0.9996954202651978}


In [ ]:
import json
torch.save(model.state_dict(), "modeloTransformer.pth")
with open("vocabulary.json", "w", encoding="utf-8") as f:
    json.dump(vocabulary, f, ensure_ascii=False, indent=4)
with open("classes.json", "w", encoding="utf-8") as f:
    json.dump(classes, f, ensure_ascii=False, indent=4)


In [ ]:
import torch

with open("vocabulary.json", "r", encoding="utf-8") as f:
    vocabulary = json.load(f)

with open("classes.json", "r", encoding="utf-8") as f:
    classes = {
        int(k): v
        for k, v in json.load(f).items()
    }

model = WordClassifierTransformer()
state_dict = torch.load(
    "modelo.pth",
    map_location="cpu",
    weights_only=True
)
model.load_state_dict(state_dict)
model.eval()

with torch.inference_mode():
    output = model(x)
    probabilities = torch.softmax(output, dim=1)
    prediction = torch.argmax(probabilities, dim=1)